In [2]:
import os
import pandas as pd
from pathlib import Path
import numpy as np
from collections import Counter
from scipy.stats import entropy

In [3]:
LABELS = ["positive", "negative", "neutral"]

In [4]:
def compute_shannon_entropy_all(base_dir: str = "data/per_entity_llm") -> pd.DataFrame:
    """
    Compute a single Shannon entropy value per model folder by aggregating
    all predicted sentiments across every entity in that folder.

    Args:
        base_dir: Base directory containing model folders

    Returns:
        DataFrame with one row per model: sentiment counts, total, and entropy
    """
    base_path = Path(base_dir)
    if not base_path.exists():
        raise FileNotFoundError(f"Base directory not found: {base_path}")

    model_folders = sorted([d for d in base_path.iterdir() if d.is_dir()])
    if not model_folders:
        print("No model folders found.")
        return pd.DataFrame()

    results = []

    for folder_path in model_folders:
        all_labels = []
        for csv_file in folder_path.glob("*.csv"):
            df = pd.read_csv(csv_file)
            if "predicted_sentiment" not in df.columns:
                continue
            all_labels.extend(df["predicted_sentiment"].dropna().tolist())

        if not all_labels:
            continue

        counts = Counter(all_labels)
        total = len(all_labels)
        probs = np.array([counts.get(l, 0) / total for l in LABELS])
        probs = probs[probs > 0]

        results.append({
            "model": folder_path.name,
            "positive": counts.get("positive", 0),
            "negative": counts.get("negative", 0),
            "neutral": counts.get("neutral", 0),
            "total": total,
            "entropy": float(entropy(probs)) if len(probs) else 0.0,
        })

    results_df = pd.DataFrame(results).sort_values("entropy", ascending=False).reset_index(drop=True)

    print("Shannon Entropy per Model (aggregated across all entities)")
    print("=" * 70)
    for _, row in results_df.iterrows():
        print(f"  {row['model']:30s} | +{row['positive']:5d}  -{row['negative']:5d}  ~{row['neutral']:5d} | H = {row['entropy']:.4f} nats")
    print("=" * 70)

    return results_df

In [5]:
def compute_shannon_entropy(folder_name: str, base_dir: str = "data/per_entity_llm") -> pd.DataFrame:
    """
    Compute Shannon entropy for each entity (person) in a folder.

    Args:
        folder_name: Name of the model folder (e.g. 'llama-3-1-8b')
        base_dir: Base directory containing model folders

    Returns:
        DataFrame with entity name, sentiment counts, and entropy
    """
    folder_path = Path(base_dir) / folder_name
    if not folder_path.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    results = []
    for csv_file in sorted(folder_path.glob("*.csv")):
        df = pd.read_csv(csv_file)
        if "predicted_sentiment" not in df.columns:
            continue
        labels = df["predicted_sentiment"].dropna().tolist()
        if not labels:
            continue

        counts = Counter(labels)
        total = len(labels)
        probs = np.array([counts.get(l, 0) / total for l in LABELS])
        probs = probs[probs > 0]

        entity_name = csv_file.stem
        results.append({
            "entity": entity_name,
            "positive": counts.get("positive", 0),
            "negative": counts.get("negative", 0),
            "neutral": counts.get("neutral", 0),
            "total": total,
            "entropy": float(entropy(probs)) if len(probs) else 0.0,
        })

    results_df = pd.DataFrame(results).sort_values("entropy", ascending=False).reset_index(drop=True)
    print(f"Folder: {folder_name} | Entities: {len(results_df)} | Mean entropy: {results_df['entropy'].mean():.4f} nats")
    return results_df

In [7]:
df_llamas = pd.DataFrame(compute_shannon_entropy_all())
df_llamas

Shannon Entropy per Model (aggregated across all entities)
  openai_gpt-4o-mini             | + 4979  - 4980  ~ 4981 | H = 1.0986 nats
  qwen_qwen2-5-vl-32b-instruct   | + 4969  - 4980  ~ 4991 | H = 1.0986 nats
  llama-3-1-8b                   | + 5175  - 5217  ~ 5208 | H = 1.0986 nats
  qwen-2-5-7b                    | + 4738  - 4930  ~ 5266 | H = 1.0977 nats
  mistralai_ministral-8b-2512    | + 4072  - 4950  ~ 5918 | H = 1.0871 nats
  openai_gpt-3-5-turbo           | + 5967  - 4980  ~ 3993 | H = 1.0854 nats
  mistralai_ministral-3b         | + 3044  - 6204  ~ 5692 | H = 1.0567 nats


,model,positive,negative,neutral,total,entropy
0,openai_gpt-4o-mini,4979,4980,4981,14940,1.098612
1,qwen_qwen2-5-vl-32b-instruct,4969,4980,4991,14940,1.098611
2,llama-3-1-8b,5175,5217,5208,15600,1.098606
3,qwen-2-5-7b,4738,4930,5266,14940,1.097656
4,mistralai_ministral-8b-2512,4072,4950,5918,14940,1.087121
5,openai_gpt-3-5-turbo,5967,4980,3993,14940,1.085432
6,mistralai_ministral-3b,3044,6204,5692,14940,1.056737


In [8]:
df_llama = pd.DataFrame(compute_shannon_entropy("llama-3-1-8b", base_dir="data/per_entity_llm"))
df_llama.value_counts("entropy")

Folder: llama-3-1-8b | Entities: 260 | Mean entropy: 1.0983 nats


entropy
1.098612    240
1.097779     12
1.095273      3
1.097779      2
1.092636      2
1.043968      1
Name: count, dtype: int64

In [11]:
df_mistral_3b = pd.DataFrame(compute_shannon_entropy("mistralai_ministral-3b", base_dir="data/per_entity_llm"))
df_mistral_3b

Folder: mistralai_ministral-3b | Entities: 249 | Mean entropy: 1.0380 nats


,entity,positive,negative,neutral,total,entropy
0,Bernardo_Arévalo,20,20,20,60,1.098612
1,Ngozi_Okonjo-Iweala,20,20,20,60,1.098612
2,Jagmeet_Singh,19,21,20,60,1.097779
3,Keir_Starmer,19,21,20,60,1.097779
4,Luis_Montenegro,19,21,20,60,1.097779
...,...,...,...,...,...,...
244,Mahatma_Gandhi,5,35,20,60,0.887694
245,Manmohan_Singh,5,35,20,60,0.887694
246,Indira_Gandhi,4,33,23,60,0.876906
247,Robert_Mugabe,4,37,19,60,0.842787


In [12]:
df_mistral_8b = pd.DataFrame(compute_shannon_entropy("mistralai_ministral-8b-2512", base_dir="data/per_entity_llm"))
df_mistral_8b

Folder: mistralai_ministral-8b-2512 | Entities: 249 | Mean entropy: 1.0753 nats


,entity,positive,negative,neutral,total,entropy
0,Mia_Mottley,20,20,20,60,1.098612
1,Aung_San_Suu_Kyi,20,20,20,60,1.098612
2,Ukhnaagiin_Khürelsükh,20,20,20,60,1.098612
3,Netumbo_Nandi-Ndaitwah,20,20,20,60,1.098612
4,Ngozi_Okonjo-Iweala,20,20,20,60,1.098612
...,...,...,...,...,...,...
244,Ronald_Reagan,6,19,35,60,0.908810
245,George_W._Bush,5,20,35,60,0.887694
246,Mohammed_bin_Salman,5,20,35,60,0.887694
247,Margaret_Thatcher,4,19,37,60,0.842787


In [20]:
df_mistral_8b[df_mistral_8b["entity"] == "Joe_Biden"]

,entity,positive,negative,neutral,total,entropy
166,Joe_Biden,16,20,24,60,1.085189


In [13]:
df_gpt_3 = pd.DataFrame(compute_shannon_entropy("openai_gpt-3-5-turbo", base_dir="data/per_entity_llm"))
df_gpt_3

Folder: openai_gpt-3-5-turbo | Entities: 249 | Mean entropy: 1.0803 nats


,entity,positive,negative,neutral,total,entropy
0,Jair_Bolsonaro,20,20,20,60,1.098612
1,Min_Aung_Hlaing,20,20,20,60,1.098612
2,Silvio_Berlusconi,20,20,20,60,1.098612
3,Ayatollah_Ali_Khamenei,20,20,20,60,1.098612
4,Ferdinand_Marcos_Jr.,20,20,20,60,1.098612
...,...,...,...,...,...,...
244,Václav_Havel,32,20,8,60,0.970116
245,John_F._Kennedy,32,20,8,60,0.970116
246,Winston_Churchill,33,20,7,60,0.945665
247,Mahatma_Gandhi,35,20,5,60,0.887694


In [ ]:
df_gpt_4 = pd.DataFrame(compute_shannon_entropy("openai_gpt-4o-mini", base_dir="data/per_entity_llm"))
df_gpt_4

Folder: openai_gpt-4o-mini | Entities: 249 | Mean entropy: 1.0986 nats


,entity,positive,negative,neutral,total,entropy
0,Abdel_Fattah_el-Sisi,20,20,20,60,1.098612
1,Ngozi_Okonjo-Iweala,20,20,20,60,1.098612
2,Min_Aung_Hlaing,20,20,20,60,1.098612
3,Mitt_Romney,20,20,20,60,1.098612
4,Mohamed_Muizzu,20,20,20,60,1.098612
...,...,...,...,...,...,...
244,Hillary_Clinton,20,20,20,60,1.098612
245,Ho_Chi_Minh,20,20,20,60,1.098612
246,Horacio_Cartes,20,20,20,60,1.098612
247,Yoweri_Museveni,20,20,20,60,1.098612


In [18]:
df_gpt_4[df_gpt_4["entity"] == "Donald_Trump"]

,entity,positive,negative,neutral,total,entropy
143,Donald_Trump,20,20,20,60,1.098612


In [17]:
df_qwen_32 = pd.DataFrame(compute_shannon_entropy("qwen_qwen2-5-vl-32b-instruct", base_dir="data/per_entity_llm"))
df_qwen_32

Folder: qwen_qwen2-5-vl-32b-instruct | Entities: 249 | Mean entropy: 1.0985 nats


,entity,positive,negative,neutral,total,entropy
0,Abdel_Fattah_el-Sisi,20,20,20,60,1.098612
1,Milos_Zeman,20,20,20,60,1.098612
2,Mitt_Romney,20,20,20,60,1.098612
3,Mohamed_Muizzu,20,20,20,60,1.098612
4,Mohamed_VI,20,20,20,60,1.098612
...,...,...,...,...,...,...
244,Abraham_Lincoln,19,20,21,60,1.097779
245,Margaret_Thatcher,18,20,22,60,1.095273
246,Fidel_Castro,18,20,22,60,1.095273
247,Kim_Jong_Un,18,20,22,60,1.095273


In [24]:
df_qwen_7 = pd.DataFrame(compute_shannon_entropy("qwen-2-5-7b", base_dir="data/per_entity_llm"))
df_qwen_7

Folder: qwen-2-5-7b | Entities: 249 | Mean entropy: 1.0955 nats


,entity,positive,negative,neutral,total,entropy
0,Shehbaz_Sharif,20,20,20,60,1.098612
1,Federico_Gutiérrez,20,20,20,60,1.098612
2,Janja_Lula_da_Silva,20,20,20,60,1.098612
3,Jagmeet_Singh,20,20,20,60,1.098612
4,Iván_Duque,20,20,20,60,1.098612
...,...,...,...,...,...,...
244,Recep_Tayyip_Erdoğan,14,20,26,60,1.068145
245,Viktor_Orbán,14,20,26,60,1.068145
246,Eamon_de_Valera,18,15,27,60,1.067094
247,Silvio_Berlusconi,13,18,29,60,1.043968
